In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# Set random seed for reproducibility
SEED = 2101
np.random.seed(SEED)
# Load the data
data = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv')
scoring_data = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv')

In [3]:
# Separate features (X) and target variable (y)
X = data.drop(columns=['monthly_spend', 'customer_id'])  # Drop target and customer_id
y = data['monthly_spend']

In [4]:
# Step 2: Preprocess Data
# Identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Impute missing values for numerical and categorical columns using SimpleImputer
numerical_imputer = SimpleImputer(strategy='mean')
categorical_imputer = SimpleImputer(strategy='most_frequent')

# Apply one-hot encoding to categorical columns
onehot_encoder = OneHotEncoder(drop='first')  # Drop first to avoid dummy variable trap

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_imputer, numerical_cols),
        ('cat', Pipeline([
            ('imputer', categorical_imputer), 
            ('onehot', onehot_encoder)
        ]), categorical_cols)
    ])

In [5]:
# Step 3: Create a Pipeline with Preprocessing and Ridge Regression
ridge_model = Ridge(random_state=2101)

# Create a pipeline that first preprocesses data, then applies Ridge regression
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ridge_model)
])

In [6]:
# Step 4: Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2101)

In [7]:
%%time
# Step 5: Hyperparameter Tuning with GridSearchCV for Ridge Regularization (alpha)
param_grid = {'regressor__alpha': np.logspace(-6, 6, 13)}  # Search over a range of alpha values
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best alpha found through grid search
best_alpha = grid_search.best_params_['regressor__alpha']
print(f"Best Alpha for Ridge Regression: {best_alpha}")

Best Alpha for Ridge Regression: 10.0
CPU times: user 542 ms, sys: 141 ms, total: 683 ms
Wall time: 3.04 s


In [9]:
%%time
# Step 6: Train the final Ridge model using the best alpha
final_ridge_model = grid_search.best_estimator_

CPU times: user 4 μs, sys: 5 μs, total: 9 μs
Wall time: 15 μs


In [10]:
# Step 7: Evaluate the final model
y_train_pred = final_ridge_model.predict(X_train)
y_test_pred = final_ridge_model.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Train RMSE (Ridge Regression): {train_rmse}, Test RMSE (Ridge Regression): {test_rmse}")

Train RMSE (Ridge Regression): 258.15118755236443, Test RMSE (Ridge Regression): 254.46264632610627


In [11]:
# Step 8: Prepare the Scoring Data and Apply the Same Transformations
# Preprocess scoring data using the same preprocessing steps
scoring_predictions = final_ridge_model.predict(scoring_data)

In [12]:
# Step 9: Create Submission File
submission_file = pd.DataFrame({
    'customer_id': scoring_data['customer_id'],
    'monthly_spend': scoring_predictions
})

# Save the submission file
submission_file.to_csv('submission_file_ridge.csv', index=False)
print("Submission file created: submission_file_ridge.csv")

Submission file created: submission_file_ridge.csv
